In [3]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import time

# Cargar modelo 
rf = joblib.load("gestovoz_rf.pkl")
print("✅ Modelo cargado")

# Config 
GESTURE_MAP = {
    0: "call",
    1: "dislike",
    2: "fist",
    3: "like",
    4: "ok",
    5: "one",
    6: "palm",
    7: "peace",
    8: "rock",
}
CONFIDENCE_THR = 0.80
ALERT_CLASS    = 6

mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils


def normalize_landmarks_xy(hand_landmarks) -> np.ndarray:
    raw_x = [lm.x for lm in hand_landmarks.landmark]
    raw_y = [lm.y for lm in hand_landmarks.landmark]
    wx, wy = raw_x[0], raw_y[0]
    feats = []
    for x, y in zip(raw_x, raw_y):
        feats.append(x - wx)
        feats.append(y - wy)
    return np.array(feats, dtype=np.float32).reshape(1, -1)


# Loop de camara 
cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode        = False,
    max_num_hands            = 1,
    min_detection_confidence = 0.60,
    min_tracking_confidence  = 0.50,
) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame  = cv2.flip(frame, 1)
        h, w   = frame.shape[:2]
        rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        # Panel de fondo 
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 80), (20, 20, 20), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

        if result.multi_hand_landmarks:
            hl = result.multi_hand_landmarks[0]

            # Dibujar landmarks
            mp_drawing.draw_landmarks(
                frame, hl, mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 180), thickness=2, circle_radius=3),
                mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2),
            )

            # Clasificar
            feats      = normalize_landmarks_xy(hl)
            pred_class = rf.predict(feats)[0]
            proba      = rf.predict_proba(feats)[0]
            confidence = proba[pred_class]
            gesture    = GESTURE_MAP[pred_class]

            over_thr   = confidence >= CONFIDENCE_THR
            is_alert   = pred_class == ALERT_CLASS

            # Colores segun estado
            if is_alert:
                color = (0, 80, 255)      # rojo → alerta critica
            elif over_thr:
                color = (0, 220, 100)     # verde → deteccion valida
            else:
                color = (0, 165, 255)     # naranja → baja confianza

            # Texto principal 
            label = f"{gesture.upper()}  {confidence:.0%}"
            cv2.putText(frame, label, (16, 52),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, color, 3, cv2.LINE_AA)

            # Barra de confianza 
            bar_w = int((w - 32) * confidence)
            cv2.rectangle(frame, (16, 62), (16 + bar_w, 72), color, -1)
            cv2.rectangle(frame, (16, 62), (w - 16, 72), (180, 180, 180), 1)

            # Badge alerta 
            if is_alert and over_thr:
                cv2.rectangle(frame, (w - 180, 90), (w - 10, 130), (0, 0, 200), -1)
                cv2.putText(frame, "ALERTA", (w - 170, 120),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

            # Top 3 probabilidades (esquina inferior) 
            sorted_p = sorted(enumerate(proba), key=lambda x: -x[1])[:3]
            for rank, (gid, p) in enumerate(sorted_p):
                txt = f"{GESTURE_MAP[gid]:<10s} {p:.2f}"
                cv2.putText(frame, txt, (16, h - 70 + rank * 24),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                            (255, 255, 255) if gid == pred_class else (140, 140, 140),
                            1, cv2.LINE_AA)

        else:
            cv2.putText(frame, "Sin mano detectada", (16, 52),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (100, 100, 100), 2)

        # FPS 
        cv2.putText(frame, f"Q: salir", (w - 100, h - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1)

        cv2.imshow("Modelo", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("✅ Camara cerrada")

✅ Modelo cargado
✅ Camara cerrada


[ WARN:0@36.827] global cap_v4l.cpp:913 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@36.828] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
E0000 00:00:1774234329.613181      20 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1774234329.613206      20 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1774234329.613212      20 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
